# Off-policy Evaluation & A/B


Throughout REC:03–REC:07 we used a *time-based split* of MovieLens and computed Recall@K / NDCG@K over the holdout. That is an *offline* metric — it is what we can compute on a static dataset. **In production** the data on which we want to measure our model is a stream of live impressions whose log was *produced by the very model we are replacing*, and who each have idiosyncratic preferences about what they would have done if shown a different item.

Two questions this notebook answers:

1. How do you estimate the value of a *new* candidate policy given logs from an *old* one? — the **off-policy evaluation** problem.
2. Where do you draw the line between exploring new candidates and exploiting known good ones? — **online** ε-greedy bandits.

The IPS estimator (Horvitz-Thompson, 1952; Li et al., 2010, applied to recsys) and its self-normalized variant (SNIPS) are the standing answer to (1). The Simulator + EpsilonGreedyBandit classes give a clean way to validate the estimate on synthetic data and then drive real exploration on top of a trained retriever.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")
import os; os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 110

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens, time_split
from notebooks.recsys.metrics import Metricator
from notebooks.recsys.models.classic import ALSRecommender, ContentRecommender
from notebooks.recsys.features import FeatureStore
from notebooks.recsys.models.retrieval import TwoTower, TwoTowerConfig, InBatchSoftmaxLoss, TwoTowerTrainer, Retriever
from notebooks.recsys.eval import Evaluator, Simulator, EpsilonGreedyBandit, simulate_ab, LoggedEvent


## Insufficient exploration: why naive offline metrics lie

Naive offline metrics assume that the model's *predicted* rewards for items the user was never exposed to are a fair estimate of what those rewards would have been. They are not. The training set itself was filtered by a previous system: the items recommended to the user in the past were selected for a reason. Items the previous policy gave low probability to never got a chance to be rated. An offline metric will systematically overestimate the policy that *looks like* the behavior policy.

A formal statement: let $\pi_b(a|x)$ be the (logged) probability that the behavior policy showed item $a$ to user $x$. Let $\pi_e$ be the policy we want to evaluate. We want $V(\pi_e) = \mathbb{E}_{x \sim D, a \sim \pi_e(\cdot|x)}[r(x, a)]$. We have logged data $\{x_t, a_t, r_t, p_t\}_{t=1}^n$ where $p_t = \pi_b(a_t | x_t)$. Under the support condition $\pi_b(a|x) > 0 \Rightarrow \pi_e(a|x) < \infty$, the IPS estimator:

$$\boxed{\, \widehat{V}_\mathrm{IPS}(\pi_e) = \frac{1}{n}\sum_{t=1}^{n} \frac{\pi_e(a_t|x_t)}{p_t} \, r_t \,}$$

is unbiased. Its variance is governed by the weight $w_t = \pi_e(a_t|x_t) / p_t$. If $w_t$ can blow up — when the behavior policy rarely chose actions that the new policy now wants to choose — the estimator's variance explodes. Effective Sample Size $\mathrm{ESS} = (\sum w_t)^2 / \sum w_t^2$ is the standard diagnostic; when ESS $< 30$, treat the estimate as suspect.

**Self-normalizing** SNIPS reduces variance at the cost of small bias:

$$\widehat{V}_\mathrm{SNIPS} = \frac{\sum_t w_t \, r_t}{\sum_t w_t}.$$

SNIPS divides by effective coverage $\sum_t w_t$ rather than $n$, which stabilizes the estimate when the behavior has over- or under-sampled different parts of the action space.


## A small-world simulator: validating the estimator bias-free

Before evaluating our trained Two-Tower against ALS logs, we validate the IPS machinery on a *known-truth* simulator. We define a behavior policy and a reward function; we draw logged impressions; we compute the IPS estimate of a target policy's value; and we compare it to the ground-truth value (which the simulator can give us by re-running the target).


In [ ]:
pool = list(range(1, 11))               # 10 items
n_users = 200
users = list(range(n_users))

# Each user has a single latent favorite item (their ground truth):
# ground_truth[u] in pool.
rng_truth = np.random.default_rng(0)
ground_truth = {u: int(rng_truth.integers(1, 11)) for u in users}

# Behavior policy picks each item with probability proportional to *popularity*,
# so popular items get logged exposure; rare items barely appear.
popularity = {i: 1.0 + (10 - i) for i in pool}    # item 1 most popular
def behavior(ctx):
    total = sum(popularity.values())
    dist = [(i, popularity[i] / total) for i in pool]
    return dist

# Reward = 1 if the shown item is the user's ground-truth favorite, else 0.
def reward_fn(u, i):
    return 1.0 if ground_truth[u] == i else 0.0

sim = Simulator(users=users, candidate_pool=pool, behavior_policy=behavior,
                 reward_fn=reward_fn, seed=0)
log = sim.run(exposures_per_user=50)
print(f"logged events: {len(log)}")

# Target policy: always pick the user's favorite item — impossible in reality but
# gives a sanity check the IPS estimator against the known true reward.
def target_picks_favorite(ctx):
    iid = int(ground_truth[ctx['user_id']])
    return {iid: 1.0}

# Ground-truth value of the target policy: every impression matches -> reward = 1.
true_value = 1.0

ev = Evaluator()
out = ev.evaluate(log, target_policy=target_picks_favorite)
print("IPS estimate:", round(out['ips'], 4), "  SNIPS:", round(out['snips'], 4),
      "  ESS:", round(out['ess'], 2), "  / true =", true_value)


**Observation.** IPS resolves to $\approx 1.0$ — within Monte Carlo noise of the ground truth. SNIPS matches. This is the calibration check; in production simulations the variance budget will eat us, but on simple support-consistent distributions IPS is correct.

What happens when the target policy displaces item weights — picks items 9 and 10 heavily that the behavior policy almost never logged?


In [ ]:
# Target: pick unpopular items (i = 8..10) with high probability.
def target_unpopular(ctx):
    sub = {i: 0.0 for i in pool}
    sub[8] = 1/3; sub[9] = 1/3; sub[10] = 1/3
    return sub
# Ground truth value per impression under this policy:
true_val = sum(reward_fn(u, i) for u in users for i in (8, 9, 10)) / (3 * len(users))
out = ev.evaluate(log, target_policy=target_unpopular)
print(f"true:    {true_val:.4f}")
print(f"IPS:     {out['ips']:.4f}   SNIPS:  {out['snips']:.4f}   ESS: {out['ess']:.1f}")


The two estimates are still approximately equal — these target policies do not violate support. But you may see the ESS drop substantially versus the favorite-picking target. If the behavior policy had $\pi_b(8|x) \approx 0$ we would either crash with NaN or rely entirely on the few $w \to \infty$ samples — IPS variance explodes, exactly the **insufficient exploration** problem.

In real recommendation logs this is constant: production models always looked *under* the long tail, so most items have never been logged. IPS estimates of policies that include them are high-variance and untrustworthy. That's why exploration is essential.


## An A/B test with a t-statistic CI

Two-arm A/B = simulate both policies against the same simulator over the same user set. Per-user reward delta has a roughly Normal distribution when $n$ is large; we report the 95% CI via the t-statistic. If the CI excludes 0 the difference is significant at $\alpha=0.05$. You should treat the CI's *narrowness* as the convert-set: too few users means you cannot tell a 5% lift from noise.


In [ ]:
class PopularPolicy:
    def recommend(self, user_id, k=10, exclude=None):
        cands = sorted(pool, key=lambda i: -popularity[i])
        if exclude:
            cands = [i for i in cands if i not in set(exclude)]
        return cands[:k]

class BestArmPolicy:
    def recommend(self, user_id, k=10, exclude=None):
        # Picks the ground-truth favorite if known (oracle).
        return [int(ground_truth[user_id])]+[0]*(k-1)

simulation = simulate_ab(
    policy_a=PopularPolicy(),
    policy_b=BestArmPolicy(),
    sim_users=list(range(50)),
    horizon=20,
    reward_fn=lambda u, i: 1.0 if ground_truth[u]==i else 0.0,
)
print(f"delta_b_minus_a   {simulation['delta_b_minus_a']:+.3f}")
print(f"95% CI              [{simulation['ci_low']:+.3f}, {simulation['ci_high']:+.3f}]")
print(f"n_users={simulation['n_users']}  horizon={simulation['horizon']}")


The oracle beats popular by a wide margin — and the CI excludes zero. In the real A/B you wouldn't have the oracle; you'd compare say TwoTower v1 (`PopularArmPolicy`-like) versus TwoTower v2 across users sampled at random.


## ε-greedy bandit on top of a trained retriever

Live broadcasts tie exploration to deployment. Rather than retrain under sample bias and trust offline metrics, you broadcast recipes through an ε-greedy policy: with probability $\epsilon$ explore (pick uniformly), else exploit (top-k from the base model). Senior products like YouTube, Netflix, Spotify use richer contextual bandits (LinUCB, Thompson Sampling), but ε-greedy is the baseline and Navy simple to deploy.


In [ ]:
cfg = MovieLensConfig(name="ml-100k")
ds = load_movielens(cfg)
train, val = time_split(ds.ratings, val_frac=0.2)

torch.manual_seed(0)
tt = TwoTower(TwoTowerConfig(n_users=ds.n_users, n_items=ds.n_items,
                              embedding_dim=32, hidden_dim=64,
                              n_negatives=0, epochs=3, batch_size=1024))
loss = InBatchSoftmaxLoss(n_items=ds.n_items, n_negatives=0, temperature=0.1)
TwoTowerTrainer(model=tt, loss=loss, cfg=tt.cfg).fit(train, verbose=False)
retriever = Retriever(model=tt, dataset=ds, train=train)
_ = retriever.recommend(int(val['user_id'].iloc[0]), k=10)

candidate_pool = list(ds.item_index.keys())
bandit = EpsilonGreedyBandit(candidate_pool=candidate_pool,
                             base_recommender=retriever,
                             epsilon=0.3, seed=0)
# Show examples of bandit acting on one user
print('10 evolutions:', [bandit.recommend(int(val['user_id'].iloc[0]), k=1)[0] for _ in range(10)])


Compare cumulative recall@10 over val users between the pure retriever and an ε-greedy bandit lot:


In [ ]:
metricator = Metricator(val)
pure = metricator.evaluate(retriever.recommend, k=10)
# Wrap the bandit's recommend call to fit `Metricator`'s `(user_id, k)` signature.
def bandit_rec(u, k=10):
    return bandit.recommend(u, k=k)
eps = metricator.evaluate(bandit_rec, k=10)
rows = [{"policy": "pure retriever",
         "recall@10": round(pure['recall@10'], 4),
         "ndcg@10": round(pure['ndcg@10'], 4),
         "coverage@10": round(pure['coverage@10'], 4)},
        {"policy": "ε-greedy (ε=0.3)",
         "recall@10": round(eps['recall@10'], 4),
         "ndcg@10": round(eps['ndcg@10'], 4),
         "coverage@10": round(eps['coverage@10'], 4)}]
pd.DataFrame(rows)


**Observation.** Recall@10 takes a small hit under ε-greedy — exploration means occasional random recommendations on val users. But recall is not the final metric. The exploration produces *logs* that the next training round desperately needs. The trade-off is captured quantitatively in production teams as the *cost of exploration*: the rate at which short-term engagement drops to fund long-term bias correction.


::: {.callout-warning}
The variance of IPS estimates in real production logs is large enough that you should not decide A/B failures from a single day's numbers, no matter how wide the t-CI. Two weeks is the typical guardrail; the *stop rule* for A/B tests is more important than the test statistic — peek-and-stop will otherwise inflate your false-positive rate.
:::


## Caveats and link forward

- **Doubly robust estimators** (Dudík, Langford, Li 2011) combine IPS with a reward model; if either is correct, the estimator is unbiased. Real production evaluations are always Doubly Robust nowadays. We skip the code here because the simpler SNIPS is enough to teach the concept and the variance problem is the same.
- **Bandit policies on top of a trained model** update too slowly to capture rapid catalog changes. A linear model + LinUCB (Li et al. 2010) over feature-space is the standard production choice.
- The /v1/feedback endpoint in REC:07 already logs click events. With a reward function and a target policy one could imitate REC:08's estimator directly from the production log captured in REC:07.

Next: REC:09 wires everything into a single Flet desktop app — the *Recommendation system demo* that brings together onboarding, retrieval + ranking + LLM re-ranking, feedback capture, and metric tracking — to close the loop.
